# 05 两比特误差敏感性评估（5 月）

目标：系统评估不同噪声机制下保真度指标的敏感性，并可视化。


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'examples' / 'backend.yaml').exists():
            return p
    raise FileNotFoundError('???????????? pyproject.toml ? examples/backend.yaml?')

ROOT = find_project_root(Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from qsim.ui.notebook import run_workflow

BACKEND_PATH = ROOT / 'examples' / 'backend.yaml'
OUT_ROOT = ROOT / 'runs' / 'roadmap_2026H1'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('ROOT =', ROOT)
print('BACKEND_PATH =', BACKEND_PATH)
print('OUT_ROOT =', OUT_ROOT)


def run_case(tag: str, qasm_text: str, hardware: dict | None = None, noise: dict | None = None, engine: str = 'qutip'):
    out_dir = OUT_ROOT / tag
    return run_workflow(
        qasm_text=qasm_text,
        backend_path=str(BACKEND_PATH),
        out_dir=str(out_dir),
        hardware=hardware or {},
        noise=noise or {},
        engine=engine,
        persist_artifacts=True,
        export_dxf=False,
    )


def get_metric(result: dict, key: str, default: float = np.nan) -> float:
    obs = result.get('analysis', {}).get('observables', {}).get('values', {})
    return float(obs.get(key, default))

In [ ]:
QASM_2Q = """
OPENQASM 3;
qubit[2] q;
bit[2] c;
h q[0];
cx q[0], q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];
"""

In [ ]:
gamma_phi_vals = [0.0, 0.002, 0.005, 0.01]
gzz_vals = [0.0, 0.005, 0.01, 0.015]
heat = np.zeros((len(gamma_phi_vals), len(gzz_vals)), dtype=float)

for i, gphi in enumerate(gamma_phi_vals):
    for j, gzz in enumerate(gzz_vals):
        hw = {'qubit_freqs_hz':[5.0e9,5.1e9], 'couplings':[{'i':0,'j':1,'g':0.02,'kind':'xx+yy'},{'i':0,'j':1,'g':gzz,'kind':'zz'}]}
        nz = {'model':'markovian_lindblad', 'gamma_phi':gphi, 't1':120.0}
        r = run_case(f'05_gphi_{gphi:.4f}_gzz_{gzz:.4f}', QASM_2Q, hardware=hw, noise=nz)
        heat[i, j] = float(r['analysis']['report']['summary'].get('fidelity_proxy', np.nan))

plt.figure(figsize=(6.8, 4.6))
plt.imshow(heat, aspect='auto', origin='lower')
plt.colorbar(label='fidelity_proxy')
plt.xticks(range(len(gzz_vals)), [f'{x:.3f}' for x in gzz_vals])
plt.yticks(range(len(gamma_phi_vals)), [f'{x:.3f}' for x in gamma_phi_vals])
plt.xlabel('g_zz'); plt.ylabel('gamma_phi'); plt.title('??????????')
plt.tight_layout(); plt.show()